In [1]:
import os
import sys

import psycopg
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from openai import OpenAI
from pymed_paperscraper import PubMed

sys.path.append(os.path.abspath(os.path.join("..")))

load_dotenv()
from langgraph.store.postgres import PostgresStore

from utils.models import ModelName, Models

In [ ]:
pubmed = PubMed(tool="langchain", email="jason@aol.com")
response = pubmed.query(query="peptides for anti-aging", max_results=5)
results = [a.toDict() for a in response]

In [7]:
with (
    psycopg.connect(os.environ["DATABASE_URl"]) as conn,
    conn.cursor() as cur,
):
    cur.execute("SELECT version();")
    print(cur.fetchone())

('PostgreSQL 18.7 (a21afb5) on aarch64-unknown-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit',)


In [11]:
with PostgresStore.from_conn_string(os.environ["DATABASE_URL"]) as store:
    store.setup()
    store.put(("test",), "ping", {"ok": True})
    print(store.get(("test",), "ping").value)
    store.delete(("test",), "ping")


{'ok': True}


In [20]:
models = Models()
llm = models.create_nvidia_model(model_name=ModelName.POOLSIDE_LAGUNA.value)

Creating model poolside/laguna-xs-2.1


d:\Bluebird\Code 1\Langchain-Foundation\.venv\Lib\site-packages\langchain_nvidia_ai_endpoints\_common.py:269: UserWarning: Found poolside/laguna-xs-2.1 in available_models, but type is unknown and inference may fail.
  warnings.warn(


In [21]:
chunks = []
for event in llm.stream("what is the correlation of bonds and crude oil prices?"):
    if isinstance(event.content, str):
        chunks.append(event.content)

response = "".join(chunks)
response


'The correlation between bonds and crude oil prices is influenced by several interconnected factors, and its behavior can vary over time depending on economic and market conditions:\n\n### 1. **Interest Rates and Inflation Expectations**  \n   - When **crude oil prices rise**, inflationary pressures often increase. To combat inflation, central banks may raise interest rates, leading to **higher yields on bonds** (lower bond prices, assuming existing bonds have lower coupons). This creates a **negative correlation** between bond prices and oil prices.  \n   - Conversely, falling oil prices can reduce inflation, prompting central banks to lower rates or maintain low yields, potentially boosting bond prices.  \n\n### 2. **Economic Growth and Flight-to-Quality**  \n   - Strong economic growth (often signaled by rising oil prices) can lead investors to favor stocks over bonds, weakening bond prices. However, if oil shocks cause economic uncertainty, investors may buy bonds as a "safe haven,

In [ ]:
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1", api_key=os.getenv("NVIDIA_API_KEy")
)
completion = client.chat.completions.create(
    model="deepseek-ai/deepseek-v4.1-flash",
    messages=[
        {
            "role": "user",
            "content": "what is the correlation between silver and gold prices and oil prices?",
        }
    ],
    temperature=1.0,
    max_tokens=262144,
)


In [ ]:
print(completion.choices[0].message.reasoning_content)


In [3]:
client = ChatNVIDIA(
    model="meta/muse-glimmer-30b",
    api_key=os.getenv("NVIDIA_API_KEY"),
    temperature=0.5,
    top_p=1,
    max_completion_tokens=262144,
)

invoke_options = {
    "frequency_penalty": 0,
    "presence_penalty": 0,
    "seed": None,
    "stop": None,
    "tool_choice": None,
    "reasoning_effort": "high",
}


In [4]:
chunks = []
for event in client.stream(
    "what is the correlation between silver and gold prices and oil prices?",
):
    if isinstance(event.content, str):
        chunks.append(event.content)

response = "".join(chunks)
print(response)


Gold and silver move together much more tightly than either of them moves with oil, and all three correlations are regime dependent.

**Gold vs Silver**
Very high and stable. Both are precious metals and are driven by the same macro factors: real interest rates, USD strength, inflation expectations and risk sentiment.
Typical long term correlation is ~0.75 to 0.90, often 0.85+ on a monthly basis. Silver is more volatile and has an industrial component so it has a higher beta to gold than gold has to silver.

**Gold vs Oil**
Positive but low to moderate, and time varying.
Long term 1970s-present the correlation is around +0.3 to +0.5. On a rolling 12 month basis it is often +0.20 to +0.45.
The link is mainly indirect: both are priced in USD and both tend to rise with inflation expectations and geopolitical risk. When the USD weakens or inflation fears rise, gold and oil tend to go up together. When oil rises for supply reasons but inflation is contained, or when real rates rise sharply,